### Hi I am using this notebook to IPL Match Information Retrieval System.

### The goal is:

#### User asks a question → ChromaDB finds the most relevant match records → returns the top matching documents.

In [38]:
import pandas as pd
import numpy as np
import ast
from dotenv import load_dotenv
import os


# =====================================================================
# DATA CLEANING LOGIC 
# =====================================================================

In [39]:
def clean_ipl_dataset(df):
    """
    Standardizes and cleans raw IPL match metrics, handling franchise
    name changes, Super Over anomalies, and parsing squads.
    """

    cleaned_df = df.copy()

    # Standardize Franchise Names.
    # FIX: added short-code aliases (CSK, MI, RCB, ...) because a lot of IPL
    # datasets store abbreviations instead of full names. Before, "CSK" was
    # never touched by this map, so the narrative text (which does expand
    # abbreviations) and the metadata (which didn't) ended up disagreeing.
    team_mapping = {
        'Delhi Daredevils': 'Delhi Capitals',
        'Kings XI Punjab': 'Punjab Kings',
        'Deccan Chargers': 'Sunrisers Hyderabad',
        'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
        'CSK': 'Chennai Super Kings', 'MI': 'Mumbai Indians',
        'RCB': 'Royal Challengers Bengaluru', 'KKR': 'Kolkata Knight Riders',
        'SRH': 'Sunrisers Hyderabad', 'DC': 'Delhi Capitals',
        'PBKS': 'Punjab Kings', 'KXIP': 'Punjab Kings', 'PK': 'Punjab Kings',
        'RR': 'Rajasthan Royals', 'GT': 'Gujarat Titans', 'LSG': 'Lucknow Super Giants'
    }

    team_cols = ['team1', 'team2', 'toss_winner', 'winner']
    for col in team_cols:
        if col in cleaned_df.columns:
            cleaned_df[col] = cleaned_df[col].astype(str).str.strip()
            cleaned_df[col] = cleaned_df[col].replace(team_mapping)

    # Handle Super Over Flags & Margins
    # FIX: always create 'is_super_over' (defaulting to False) even if the
    # 'super_over' column is missing, so later code that reads this column
    # never breaks.
    if 'super_over' in cleaned_df.columns:
        cleaned_df['is_super_over'] = cleaned_df['super_over'].astype(str).str.upper().str.strip().isin(['Y', 'YES', '1', 'TRUE'])
    else:
        cleaned_df['is_super_over'] = False

    # Normalize and Cross-Validate Margins
    if 'win_by_runs' in cleaned_df.columns and 'win_by_wickets' in cleaned_df.columns:
        cleaned_df['win_by_runs'] = pd.to_numeric(cleaned_df['win_by_runs'], errors='coerce').fillna(0).astype(int)
        cleaned_df['win_by_wickets'] = pd.to_numeric(cleaned_df['win_by_wickets'], errors='coerce').fillna(0).astype(int)

        cleaned_df.loc[cleaned_df['is_super_over'], ['win_by_runs', 'win_by_wickets']] = 0

        cleaned_df['win_type'] = np.where(cleaned_df['win_by_runs'] > 0, 'Runs',
                                  np.where(cleaned_df['win_by_wickets'] > 0, 'Wickets',
                                  np.where(cleaned_df['is_super_over'], 'Super Over', 'No Result')))

        cleaned_df.loc[cleaned_df['win_type'] == 'Runs', 'win_by_wickets'] = 0
        cleaned_df.loc[cleaned_df['win_type'] == 'Wickets', 'win_by_runs'] = 0

    # FIX: you mentioned target_runs / target_overs in your dataset, but the
    # original code never touched them. Coerce them to numeric so they don't
    # silently sit around as messy strings.
    for col in ['target_runs', 'target_overs']:
        if col in cleaned_df.columns:
            cleaned_df[col] = pd.to_numeric(cleaned_df[col], errors='coerce')

    # Parse String Arrays into clean Python lists
    def clean_playing_xi(squad_data):
        # FIX: pd.isna() throws on a list/array input. Guard for that first.
        if not isinstance(squad_data, list) and pd.isna(squad_data):
            return []
        if isinstance(squad_data, str):
            squad_data = squad_data.strip()
            if squad_data.startswith('[') and squad_data.endswith(']'):
                try:
                    squad_data = ast.literal_eval(squad_data)
                except (ValueError, SyntaxError):
                    squad_data = squad_data.replace('[', '').replace(']', '').split(',')
            else:
                squad_data = squad_data.split(',')

        if isinstance(squad_data, list):
            cleaned_squad = []
            for player in squad_data:
                p_clean = str(player).strip()
                for suffix in ['(c)', '(wk)', ' (c)', ' (wk)', '(sub)']:
                    p_clean = p_clean.replace(suffix, '')
                p_clean = p_clean.strip()
                if p_clean:
                    cleaned_squad.append(p_clean)
            return cleaned_squad
        return []

    if 'playing_xi' in cleaned_df.columns:
        cleaned_df['playing_xi'] = cleaned_df['playing_xi'].apply(clean_playing_xi)

    return cleaned_df


# =====================================================================
# VECTOR DB SERIALIZATION LOGIC 
# =====================================================================

In [40]:
def serialize_row_for_chromadb(row):
    """
    Takes a single CLEANED row and returns a tuple: (narrative_text, metadata_dict)
    """

    # FIX: a tiny helper that treats NaN as "missing" even when the column
    # exists. row.get('x', default) does NOT protect you from NaN values
    # that ARE present in the column -- .get() only falls back to default
    # when the KEY is missing entirely, not when the value is empty/NaN.
    def safe(val, default):
        try:
            if pd.isna(val):
                return default
        except (TypeError, ValueError):
            pass
        return val

    season = row.get('season', 'Unknown Season')
    t1 = safe(row.get('team1'), 'Team 1')
    t2 = safe(row.get('team2'), 'Team 2')
    winner = safe(row.get('winner'), 'No Result')
    toss_winner = safe(row.get('toss_winner'), 'Unknown')
    toss_decision = str(safe(row.get('toss_decision'), '')).lower()
    venue = safe(row.get('venue'), 'an unknown stadium')
    pom = safe(row.get('player_of_the_match'), 'No one (no result)')
    result = safe(row.get('result'), 'Match completed')
    # FIX: you said your dataset has playoff stages (Qualifier 1, Qualifier 2,
    # Eliminator, Final) -- the old code never captured this at all. If your
    # real data has a column like 'match_type' or 'stage', this will pick it
    # up automatically; otherwise it defaults to "League".
    match_type = safe(row.get('match_type', row.get('stage')), 'League')

    # By this point team1/team2/winner/toss_winner are ALREADY standardized
    # full names (clean_ipl_dataset did that), so we no longer need a
    # separate abbreviation_map here -- that removes the old inconsistency
    # where narrative text used full names but metadata used raw codes.

    squad = row.get('playing_xi', [])
    squad_text = f" The playing XI included: {', '.join(squad)}." if isinstance(squad, list) and squad else ""

    narrative_text = (
        f"In the {season} IPL season, a {match_type} match was played between {t1} and {t2} at the {venue} stadium. "
        f"{toss_winner} won the toss and elected to {toss_decision} first. "
        f"The final outcome was: {result}. The official winner of the match was {winner}. "
        f"For a standout performance, {pom} was awarded the Player of the Match.{squad_text}"
    )

    # FIX: metadata now uses the SAME standardized values as the narrative
    # (t1/t2/winner/toss_winner), and adds several fields you'll want for
    # filtering later: match_type (for playoffs), venue, win_type,
    # win_by_runs/win_by_wickets, is_super_over.
    metadata = {
        "season": int(season) if str(season).isdigit() else 0,
        "team1": str(t1),
        "team2": str(t2),
        "winner": str(winner),
        "toss_winner": str(toss_winner),
        "match_type": str(match_type),
        "venue": str(venue),
        "player_of_the_match": str(pom),
        "win_type": str(row.get('win_type', 'Unknown')),
        "win_by_runs": int(row.get('win_by_runs', 0)),
        "win_by_wickets": int(row.get('win_by_wickets', 0)),
        "is_super_over": bool(row.get('is_super_over', False)),
    }

    return pd.Series([narrative_text, metadata])


# =====================================================================
# THE UNIFIED EXECUTION PIPELINE
# =====================================================================

In [41]:
if __name__ == "__main__":
    # Mocking your raw dirty data inputs
    raw_match_data = {
        'season': [2024],
        'team1': ['CSK'],
        'team2': ['MI'],
        'winner': ['CSK '],  # Intentionally trailing space
        'toss_winner': ['MI'],
        'toss_decision': ['Bowl'],
        'venue': ['Chennai'],
        'player_of_the_match': ['Jadeja'],
        'result': ['CSK won by 6 wickets'],
        'win_by_runs': ["0"],       # Intentionally string type
        'win_by_wickets': [6],
        'super_over': ['N'],
        'playing_xi': ["['MS Dhoni (wk)', 'Jadeja (c)', 'Ruturaj Gaikwad']"]
    }

    df_raw = pd.DataFrame(raw_match_data)

    print("--- Executing Combined Pipeline ---")

    df_cleaned = clean_ipl_dataset(df_raw)
    df_cleaned[['chroma_document', 'chroma_metadata']] = df_cleaned.apply(serialize_row_for_chromadb, axis=1)

    print("\n[PREPARED EMBEDDING DOCUMENT]:")
    print(df_cleaned['chroma_document'].iloc[0])

    print("\n[PREPARED METADATA DICTIONARY]:")
    print(df_cleaned['chroma_metadata'].iloc[0])


--- Executing Combined Pipeline ---

[PREPARED EMBEDDING DOCUMENT]:
In the 2024 IPL season, a League match was played between Chennai Super Kings and Mumbai Indians at the Chennai stadium. Mumbai Indians won the toss and elected to bowl first. The final outcome was: CSK won by 6 wickets. The official winner of the match was Chennai Super Kings. For a standout performance, Jadeja was awarded the Player of the Match. The playing XI included: MS Dhoni, Jadeja, Ruturaj Gaikwad.

[PREPARED METADATA DICTIONARY]:
{'season': 2024, 'team1': 'Chennai Super Kings', 'team2': 'Mumbai Indians', 'winner': 'Chennai Super Kings', 'toss_winner': 'Mumbai Indians', 'match_type': 'League', 'venue': 'Chennai', 'player_of_the_match': 'Jadeja', 'win_type': 'Wickets', 'win_by_runs': 0, 'win_by_wickets': 6, 'is_super_over': False}


## Loading the real 18-season data, run it through the two fixed functions.

### Step 1: Figure out what your data looks like
### First, check what you actually have — one big file with an 18-season column, or one file per season that need combining.

In [42]:
import pandas as pd
import glob

# Case A: one single file with all seasons
df_real_raw = pd.read_csv("ipl.csv")   # or pd.read_excel(...) if it's .xlsx

# # Case B: one file per season that need combining
# season_files = glob.glob("data/ipl_season_*.csv")
# df_real_raw = pd.concat([pd.read_csv(f) for f in season_files], ignore_index=True)

print(df_real_raw.shape)
df_real_raw.head()

(1169, 19)


,team1,team2,match_date,toss_winner,toss_decision,winner,player_of_match,venue,city,team1_players,team2_players,season,match_number,match_type,result,result_margin,target_runs,target_overs,super_over
0,Royal Challengers Bangalore,Kolkata Knight Riders,2008-04-18,Royal Challengers Bangalore,field,Kolkata Knight Riders,BB McCullum,M Chinnaswamy Stadium,Bangalore,"R Dravid, W Jaffer, V Kohli, JH Kallis, CL Whi...","SC Ganguly, BB McCullum, RT Ponting, DJ Hussey...",1,1,League,runs,140.0,223.0,20.0,N
1,Kings XI Punjab,Chennai Super Kings,2008-04-19,Chennai Super Kings,bat,Chennai Super Kings,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Chandigarh,"K Goel, JR Hopes, KC Sangakkara, Yuvraj Singh,...","PA Patel, ML Hayden, MEK Hussey, MS Dhoni, SK ...",1,2,League,runs,33.0,241.0,20.0,N
2,Delhi Daredevils,Rajasthan Royals,2008-04-19,Rajasthan Royals,bat,Delhi Daredevils,MF Maharoof,Feroz Shah Kotla,Delhi,"G Gambhir, V Sehwag, S Dhawan, MK Tiwary, KD K...","T Kohli, YK Pathan, SR Watson, M Kaif, DS Lehm...",1,3,League,wickets,9.0,130.0,20.0,N
3,Mumbai Indians,Royal Challengers Bangalore,2008-04-20,Mumbai Indians,bat,Royal Challengers Bangalore,MV Boucher,Wankhede Stadium,Mumbai,"L Ronchi, ST Jayasuriya, DJ Thornely, RV Uthap...","S Chanderpaul, R Dravid, LRPL Taylor, JH Kalli...",1,4,League,wickets,5.0,166.0,20.0,N
4,Kolkata Knight Riders,Deccan Chargers,2008-04-20,Deccan Chargers,bat,Kolkata Knight Riders,DJ Hussey,Eden Gardens,Kolkata,"WP Saha, BB McCullum, RT Ponting, SC Ganguly, ...","AC Gilchrist, Y Venugopal Rao, VVS Laxman, A S...",1,5,League,wickets,5.0,111.0,20.0,N


## Step 2: Check the column names line up
### Your clean_ipl_dataset and serialize_row_for_chromadb functions expect specific column names (team1, team2, winner, toss_winner, toss_decision, venue, player_of_the_match, result, win_by_runs, win_by_wickets, super_over, playing_xi, match_type/stage). Real datasets (like the popular Kaggle IPL dataset) often use slightly different names.

In [43]:
print(df_real_raw.columns.tolist())

['team1', 'team2', 'match_date', 'toss_winner', 'toss_decision', 'winner', 'player_of_match', 'venue', 'city', 'team1_players', 'team2_players', 'season', 'match_number', 'match_type', 'result', 'result_margin', 'target_runs', 'target_overs', 'super_over']


## Step 3: Run the two functions

In [44]:
df_cleaned = clean_ipl_dataset(df_real_raw)

df_cleaned[['chroma_document', 'chroma_metadata']] = df_cleaned.apply(
    serialize_row_for_chromadb, axis=1
)

# sanity check a few rows before feeding 1000+ matches into ChromaDB
print(df_cleaned['chroma_document'].iloc[0])
print(df_cleaned['chroma_metadata'].iloc[0])
print(df_cleaned['chroma_document'].sample(3).tolist())

In the 1 IPL season, a League match was played between Royal Challengers Bengaluru and Kolkata Knight Riders at the M Chinnaswamy Stadium stadium. Royal Challengers Bengaluru won the toss and elected to field first. The final outcome was: runs. The official winner of the match was Kolkata Knight Riders. For a standout performance, No one (no result) was awarded the Player of the Match.
{'season': 1, 'team1': 'Royal Challengers Bengaluru', 'team2': 'Kolkata Knight Riders', 'winner': 'Kolkata Knight Riders', 'toss_winner': 'Royal Challengers Bengaluru', 'match_type': 'League', 'venue': 'M Chinnaswamy Stadium', 'player_of_the_match': 'No one (no result)', 'win_type': 'Unknown', 'win_by_runs': 0, 'win_by_wickets': 0, 'is_super_over': False}
['In the 5 IPL season, a League match was played between Punjab Kings and Sunrisers Hyderabad at the Punjab Cricket Association Stadium, Mohali stadium. Sunrisers Hyderabad won the toss and elected to bat first. The final outcome was: wickets. The offic

## Step 4: Look for gaps before moving on
### Real data is messy — check for anything that would silently produce a bad narrative:

In [45]:
print(df_cleaned[['team1', 'team2', 'winner']].isna().sum())
print(df_cleaned['match_type'].value_counts() if 'match_type' in df_cleaned else "no match_type column found")

team1     0
team2     0
winner    0
dtype: int64
match_type
League                1099
Final                   18
Qualifier 1             15
Qualifier 2             15
Eliminator              12
Semi Final               6
Elimination Final        3
3rd Place Play-Off       1
Name: count, dtype: int64


# Create the Chroma Database

In [46]:
import chromadb
from sentence_transformers import SentenceTransformer

# Persistent database
client = chromadb.PersistentClient(path="./ipl_chromadb")

collection = client.get_or_create_collection(
    name="ipl_matches"
)

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

python-dotenv could not parse statement starting at line 1
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6420.27it/s]


# Generate Embeddings

In [47]:
documents = df_cleaned["chroma_document"].tolist()

metadatas = df_cleaned["chroma_metadata"].tolist()

ids = [f"match_{i}" for i in range(len(documents))]

embeddings = embedding_model.encode(
    documents,
    show_progress_bar=True
).tolist()

Batches: 100%|██████████| 37/37 [00:03<00:00, 11.61it/s]


# Store in ChromaDB

In [48]:
collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas
)

# Semantic Search

In [49]:
def semantic_search(query, top_k=5):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    return results

In [50]:
results = semantic_search(
    "Which matches did CSK win in Chennai?"
)

print(results)

{'ids': [['match_345', 'match_720', 'match_394', 'match_384', 'match_754']], 'embeddings': None, 'documents': [['In the 6 IPL season, a League match was played between Delhi Capitals and Chennai Super Kings at the Feroz Shah Kotla stadium. Chennai Super Kings won the toss and elected to bat first. The final outcome was: runs. The official winner of the match was Chennai Super Kings. For a standout performance, No one (no result) was awarded the Player of the Match.', 'In the 12 IPL season, a League match was played between Rajasthan Royals and Chennai Super Kings at the Sawai Mansingh Stadium stadium. Chennai Super Kings won the toss and elected to field first. The final outcome was: wickets. The official winner of the match was Chennai Super Kings. For a standout performance, No one (no result) was awarded the Player of the Match.', 'In the 6 IPL season, a Qualifier 1 match was played between Chennai Super Kings and Mumbai Indians at the Feroz Shah Kotla stadium. Chennai Super Kings w

# Build Context

In [51]:
def build_context(results):

    docs = results["documents"][0]

    context = "\n\n".join(docs)

    return context

#  Ask the LLM


In [54]:
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("API_KEY"),
    base_url="https://api.cerebras.ai/v1"
)

In [55]:
def answer_question(question):

    results = semantic_search(question)

    context = build_context(results)

    prompt = f"""
You are an IPL expert.

Answer ONLY using the information below.

Context:

{context}

Question:

{question}

If the answer is not available, say:
'I couldn't find that information in the IPL database.'
"""

    response = client.chat.completions.create(
    model="gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=1.7
)
    
    answer = response.choices[0].message.content

    print(answer)

In [60]:
answer_question(
    "Who won the max number of matches in the 2024 IPL season?"
)

I couldn't find that information in the IPL database.
